<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_32_file_level_release_package.ipynb">9.32 文件级示例仓库与发布包</a></li>
        <li>下一节： <a href="9_34_processing_report_review_rubric.ipynb">9.34 处理报告与复查量规</a></li>
    </ul>
</div>


## 9.33 轻量样本数据集与练习包：从真实观测到可分发材料

射电干涉测量的真实数据往往很大。完整 Measurement Set、校准表、pipeline 产品、图像 cube 和日志文件很容易达到数十 GB 甚至更高。教材若只依赖完整数据，实践训练会被下载、存储和软件环境拖住；教材若只给静态图片，又无法训练处理判断。轻量样本数据集的任务，是在二者之间建立一座桥：保留科学判断所需的不变量，去掉不影响当前训练目标的体积和复杂度。

轻量样本不能被理解为“随便裁剪一小块”或“把通道平均到文件变小”。样本设计本身就是一次观测和处理问题。需要先说明该样本要训练什么判断：是检查元数据和质量报告，还是比较成像权重；是测量低面亮度上限，还是验证源表可靠性；是识别校准失败，还是复查发布包。不同目标需要保留的量不同。一个适合连续谱源表训练的样本，不一定适合最大可恢复尺度教学；一个适合低面亮度上限的样本，也不一定适合谱线速度场练习。

本节把轻量样本数据集定义为一个有科学边界的练习包。它可以来自真实归档数据的子集，也可以由真实产品和少量合成扰动组成；可以只包含图像和清单，也可以包含精简 Measurement Set 和可重跑参数。关键要求是：样本包必须说明自己能训练什么，不能代表什么，并给出验证依据。


![轻量样本数据集漏斗](figures/practical_sample_dataset_funnel.png)

图 9.33.1 把完整观测到练习包的过程画成一个漏斗。完整观测包含全部时间、频窗、天线、校准表和原始质量报告；任务子集只保留目标场、关键校准源、代表性频窗和关键时间段；压缩数据通过必要平均、裁剪区域、产品清单和校验和固定身份；最终练习包还要包含输入、参数、期望产品、失败样本和报告模板。


### 9.33.1 样本数据集的目标：保留判断结构

一个好的样本包首先要保留判断结构，而不是保留完整观测的全部细节。判断结构由科学问题决定。例如，若目标是训练低面亮度连续谱测量，样本必须保留合成束、噪声统计、主波束范围、短基线信息或至少保留能够证明短基线限制的元数据。若目标是训练源表验证，样本必须保留局部噪声变化、残差结构、检测阈值、注入源或人工检查区域。若目标是训练 pipeline 质量报告判读，样本必须保留 weblog 摘要、flag 统计、校准解异常和相应产品。

这意味着轻量化不是一个单纯的数据工程动作，而是一个科学抽象动作。它回答的问题是：哪些对象是本练习的核心变量，哪些对象只是运行细节。核心变量一旦被删掉，练习就会失去科学意义；运行细节可以被压缩、替换或固定，只要不会改变结论边界。

一个可用的样本包说明应包含三句话。第一，样本来自什么类型的数据或产品；第二，样本保留了哪些科学不变量；第三，样本不能用于哪些结论。这样的说明比“这是一个小数据集”更有价值，因为它直接限定了练习的解释范围。


### 9.33.2 科学不变量：哪些量不能被压缩掉

样本压缩前需要先定义科学不变量。所谓不变量，不是数学上绝对不变的量，而是在当前教学任务中不能被压缩改变的判断依据。对成像参数练习而言，$uv$ 覆盖、波束、噪声和残差结构是关键不变量。对谱线练习而言，频率覆盖、通道宽度、速度约定、连续谱扣除空间和 mask 行为是关键不变量。对偏振练习而言，频率覆盖、$\lambda^2$ 采样、泄漏校准证据和 Faraday 分辨率是关键不变量。

压缩动作应围绕这些不变量设计。时间平均会改变视场边缘的响应；频率平均会引入带宽展宽；裁剪天线或基线会改变最大可恢复尺度和点扩散函数；裁剪图像会丢掉主波束边缘噪声和强源旁瓣；只保留最终图像会丢掉校准和 flag 证据。因此，样本设计必须先写出“不变量清单”，再决定怎样裁剪。


![样本科学不变量矩阵](figures/practical_sample_invariant_matrix.png)

图 9.33.2 给出一个不变量矩阵示例。频率覆盖、最短基线、主波束区域、校准证据和噪声统计通常需要保留；某些空间分辨率可以降低；部分校准证据可替换为产品摘要；失败模式若只保留成功路径，则不能代表质量判断训练。


样本说明可以写成下面的形式：

```yaml
sample_purpose: low_surface_brightness_continuum_training
preserved_invariants:
  angular_scale_range: 30arcsec_to_3arcmin
  primary_beam_region: response_above_0.3
  noise_statistics: local_rms_map_and_residual_image
  calibration_evidence: weblog_summary_and_gain_diagnostics
  product_identity: image_catalog_region_checksums
not_supported:
  - full_spectral_index_measurement
  - structures_larger_than_recorded_mrs
  - absolute_flux_scale_better_than_documented_systematics
```

这类清单让练习的边界变得清楚。若样本只保留单频段连续谱图像，它就不应被用来训练谱指数测量；若样本没有短基线信息或最大可恢复尺度记录，它就不应被用来判断大尺度非探测；若样本没有校准证据，它可以训练图像测量，却不能训练校准质量审计。


### 9.33.3 平均、裁剪与产品替代的损失预算

轻量化最常见的手段是时间平均、频率平均、空间裁剪和产品替代。每一种手段都有损失预算。时间平均和频率平均会使离相位中心较远的源发生展宽和峰值降低。近似地，带宽平均带来的幅度损失可写成 sinc 型形式：

$$
A_{\nu}(\theta) \simeq \operatorname{sinc}\left(\pi \frac{B\theta\Delta\nu}{c}\right),
$$

其中 $B$ 是投影基线长度，$\theta$ 是相对相位中心的角距离，$\Delta\nu$ 是平均后的通道宽度。时间平均的损失可以写成类似形式：

$$
A_t(\theta) \simeq \operatorname{sinc}\left(\pi\omega_{\oplus}\frac{B\theta\Delta t}{\lambda}\right),
$$

其中 $\omega_{\oplus}$ 是地球自转角速度，$\Delta t$ 是积分或平均时间。这两个式子并不是为了替代成像软件的精确计算，而是提醒样本设计必须问一个问题：平均后的数据还是否保留练习所需的视场和峰值保真度。

空间裁剪也有损失。若只裁剪目标附近的小图，强源旁瓣、主波束边缘噪声和背景估计区域可能被删掉。对于源表训练，这会让检测可靠性看起来过于乐观。对于低面亮度测量，这会让背景选择和上限估计失去上下文。产品替代也需要说明边界：用 weblog 摘要替代完整校准表可以训练质量报告判读，但不能训练重新求解校准；用图像替代 Measurement Set 可以训练通量测量和源表，但不能训练 flag 和成像参数如何改变可见度权重。


![样本裁剪与平均决策](figures/practical_sample_reduction_decision.png)

图 9.33.3 展示了裁剪和平均的决策流程。先选择科学量，再定义角尺度和频谱尺度，随后估算展宽和灵敏度损失，生成小样本，最后验证结论是否不变。若验证失败，需要减少平均、扩大裁剪范围或降低练习任务要求。


一个样本包可以把损失预算写成简短报告：平均后的视场半径内峰值损失小于多少；taper 后亮温灵敏度相对原图怎样变化；裁剪区域是否仍包含背景估计带；主波束响应最低到多少；源表可靠性验证是否仍包含边缘区域和强源附近区域。只要这些边界写清，样本即使不完整，也能成为严肃的训练材料。


### 9.33.4 真实子集、产品子集与合成扰动

轻量样本大致有三种来源。第一种是真实子集，直接从归档数据中选取目标场、少数频窗、有限时间段和必要校准源。这种样本最接近真实处理，但仍可能较大，也需要完整软件环境。第二种是产品子集，只保留图像、残差、源表、region、质量报告和参数文件。这种样本适合科学测量、源表验证、产品差异审计和限制声明训练。第三种是合成扰动，在真实产品基础上注入已知源、模拟残差结构或构造失败样本。这种样本适合训练错误识别和选择函数，但必须明确它不是独立真实观测。

三种来源可以组合。一个连续谱源表练习包可以使用真实图像和残差，加入注入源目录，保留 PyBDSF 或简化源搜索参数，再提供人工检查小样本。一个归档复盘练习包可以使用真实 weblog 摘要和产品清单，同时用合成差异图模拟不同成像参数的后果。一个校准诊断练习包可以使用真实质量报告中的异常图，配合小型模拟可见度展示相位跳变怎样进入图像残差。

选择哪一种来源，取决于训练目标。若目标是“会判断”，产品子集和失败样本往往足够；若目标是“会重跑”，需要真实子集或精简 Measurement Set；若目标是“会设计实验”，需要保留从完整观测到小样本的裁剪报告，让练习本身成为方法的一部分。


### 9.33.5 从易到难的练习层

同一个样本包可以设计成多个练习层。观察层只要求检查元数据、图像和质量报告，形成初步判断。处理层要求修改参数、局部重跑和产品对比。测量层要求计算通量、源表、上限和误差预算。审计层要求识别失败样本、写限制声明和整理发布包。这样的分层可以让同一套材料服务不同训练深度，也能避免实践训练变成一次性演示。


![样本练习层](figures/practical_sample_exercise_layers.png)

图 9.33.4 展示了从观察层、处理层、测量层到审计层的练习设计。同一输入可以对应不同任务，同一报告模板可以要求不同深度。分层练习让本科高年级到研究训练可以共用材料，同时保持清楚的能力边界。


分层设计的关键是每一层都有独立产物。观察层产物可以是质量判读表；处理层产物可以是参数差异和图像差异；测量层产物可以是通量表、源表或上限报告；审计层产物可以是限制声明、失败样本说明和发布包清单。若只有最终答案，没有中间判断，样本包就难以训练真实研究中的决策能力。


### 9.33.6 标准路径、盲测任务与失败对照

样本包还应包含三类任务。标准路径提供参考答案，用于校准期望结果和检查软件环境。盲测任务使用相同数据或相似产品，但隐藏部分判断结论，用于检验迁移能力。失败对照故意保留常见错误，例如过度平均导致视场边缘峰值降低、裁剪区域缺少背景估计带、旧源表配新图像、主波束边缘伪检未排除、最大可恢复尺度不足却写成严格非探测。

没有失败对照的样本包，很难训练质量判断；没有盲测任务的样本包，很难检验是否只是复现了标准答案。三类任务应使用统一评分依据，例如清单是否完整、参数是否可追溯、产品差异是否解释清楚、限制声明是否与证据相符。


![标准路径、盲测任务与失败对照](figures/practical_sample_validation_split.png)

图 9.33.5 中，标准路径用于校准期望结果，盲测任务用于独立判断，失败对照用于识别错误模式。三者共同回到统一评分依据：清单、参数、产品差异和限制声明。


一个源表训练样本可以这样设计。标准路径给出图像、局部噪声图、源搜索参数和参考源表。盲测任务更换检测阈值或加入一个局部噪声不均匀区域，要求重新判断可靠性。失败对照则给出一个看似源数更多的目录，但新增源集中在强源残差和主波束边缘。评分时不只看源数是否接近参考答案，而要看是否识别出选择函数变化和伪检风险。

一个低面亮度连续谱样本也可以这样设计。标准路径展示 taper 后上限计算；盲测任务改变测量区域或主波束裁剪；失败对照使用高分辨率图像直接给出弥散通量上限。评分重点不是是否得到某个数字，而是是否说明 beam、相关噪声、最大可恢复尺度和主波束误差共同限制了结论。


### 9.33.7 教学案例：归档连续谱小样本包

考虑一个面向连续谱归档复盘的轻量样本包。完整观测很大，不适合直接分发；训练目标也不是完整重校准，而是判断一个归档产品是否适合低面亮度测量和源表验证。样本包可以保留三类输入：一组原 pipeline 图像、一个 taper 重跑图像和残差图、一个质量报告摘要。再配套三个清单：数据身份清单、参数清单、产品清单。最后提供一个标准路径、一个盲测任务和两个失败对照。

标准路径要求比较原图和 taper 图，计算共同 beam 后的区域通量上限，并写出最大可恢复尺度限制。盲测任务给出另一个测量区域，要求重新估计局部噪声、主波束裁剪和上限。第一个失败对照使用原高分辨率图像直接计算低面亮度上限；第二个失败对照给出一个源表，新增源主要集中在强源残差附近。这样设计后，小样本包虽然没有完整 Measurement Set，却能训练归档复盘中最关键的判断。

该样本包的 README 应明确写出不能支持的结论。例如，它不能训练完整 flag 策略，不能验证新的校准解，不能回答谱指数问题，不能排除大于记录最大可恢复尺度的弥散结构。这样的限制并不会削弱样本价值，反而让练习更接近真实研究：任何数据都只能回答它确实保留了证据的问题。


### 9.33.8 本节结论

轻量样本数据集的核心不是压缩体积，而是保留判断结构。样本设计必须从科学问题出发，定义需要保留的频率覆盖、角尺度、主波束区域、校准证据、噪声统计和失败模式，再决定时间平均、频率平均、图像裁剪和产品替代的边界。每一次压缩都应有损失预算，每一个样本都应说明支持和不支持的结论。

第 9.33 节与前面几节共同构成可复现实践的最后一块拼图。第 9.30 节定义一次运行事件，第 9.31 节定义归档复盘证据链，第 9.32 节定义文件级发布包，第 9.33 节定义如何把真实观测转化为可分发、可练习、可审计的小样本。这样教材实践不再依赖庞大数据或静态截图，而可以形成从观察、处理、测量到审计的连续训练链。
